#01 - Data Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

train = pd.read_csv('../data/raw/train.csv')
stores = pd.read_csv('../data/raw/stores.csv')
oil = pd.read_csv('../data/raw/oil.csv')
holidays = pd.read_csv('../data/raw/holidays_events.csv')
transactions = pd.read_csv('../data/raw/transactions.csv')

print("Loaded")

In [ ]:
print(f"Shape: {train.shape}")
print(f"\nColumns: {list(train.columns)}")
train.head(10)

In [ ]:
train.info()

In [ ]:
train['date'] = pd.to_datetime(train['date'])
print(f"Από: {train['date'].min()}")
print(f"Μέχρι: {train['date'].max()}")
print(f"Διάρκεια: {(train['date'].max() - train['date'].min()).days} μέρες")

In [ ]:
print(f"Καταστήματα: {train['store_nbr'].nunique()}")
print(f"Οικογένειες προϊόντων: {train['family'].nunique()}")
print(f"\nΛίστα οικογενειών:")
print(train['family'].unique())

In [ ]:
train['sales'].describe()

In [ ]:
zero_sales = (train['sales'] == 0).sum()
total = len(train)
print(f"Γραμμές με sales=0: {zero_sales:,} από {total:,} ({100*zero_sales/total:.1f}%)")

In [ ]:
zero_by_family = train.groupby('family').apply(
    lambda x: (x['sales'] == 0).sum() / len(x) * 100
).sort_values(ascending=False)
print("Ποσοστό μηδενικών ανά οικογένεια:")
print(zero_by_family.round(1))

In [ ]:
zero_by_store = train.groupby('store_nbr').apply(
    lambda x: (x['sales'] == 0).sum() / len(x) * 100
).sort_values(ascending=False)
print("Top 10 stores με τα περισσότερα μηδενικά:")
print(zero_by_store.head(10).round(1))
print("\nBottom 10 stores με τα λιγότερα μηδενικά:")
print(zero_by_store.tail(10).round(1))

In [ ]:
# Δείγμα 20 γραμμών με μηδενικά
train[train['sales'] == 0].sample(20, random_state=42)

In [ ]:
# Για κάθε ζευγάρι (store, family), πόσο μέσο όρο πωλήσεων έχει
pair_avg = train.groupby(['store_nbr', 'family'])['sales'].mean()

# Πόσα ζευγάρια έχουν πάντα 0;
total_pairs = len(pair_avg)
always_zero = (pair_avg == 0).sum()
mostly_zero = (pair_avg < 1).sum()  # σχεδόν πάντα 0

print(f"Συνολικά (store, family) ζευγάρια: {total_pairs}")  # θα είναι 54 × 33 = 1,782
print(f"Always-zero ζευγάρια: {always_zero} ({100*always_zero/total_pairs:.1f}%)")
print(f"Σχεδόν-always-zero (avg<1): {mostly_zero} ({100*mostly_zero/total_pairs:.1f}%)")

In [ ]:
# Average sales per family για το store 52
store_52 = train[train['store_nbr'] == 52].groupby('family')['sales'].agg(['mean', 'sum'])
store_52_sorted = store_52.sort_values('sum', ascending=False)
print("Store 52 - πωλήσεις ανά οικογένεια:")
print(store_52_sorted)

In [ ]:
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Raw distribution
axes[0].hist(train['sales'], bins=100)
axes[0].set_title('Sales distribution (raw)')
axes[0].set_xlabel('Sales')
axes[0].set_ylabel('Frequency')
axes[0].set_yscale('log')  # log scale γιατί τα 0 κυριαρχούν

# Log distribution (μόνο για non-zero)
non_zero_sales = train.loc[train['sales'] > 0, 'sales']
axes[1].hist(np.log1p(non_zero_sales), bins=100)
axes[1].set_title('Sales distribution (log scale, non-zero only)')
axes[1].set_xlabel('log(1 + sales)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
for col in stores.columns:
    if stores[col].dtype == 'object' or stores[col].dtype == 'str':
        print(f"{col}: {stores[col].nunique()} μοναδικές → {sorted(stores[col].unique())}")
    elif col != 'store_nbr':
        print(f"{col}: range [{stores[col].min()}, {stores[col].max()}], {stores[col].nunique()} μοναδικές")

In [ ]:
print(f"Shape: {oil.shape}")
print(f"\nColumns: {list(oil.columns)}")
print(f"\nDtypes:")
print(oil.dtypes)
print(f"\nMissing values:")
print(oil.isnull().sum())
print(f"\nFirst rows:")
oil.head(10)

In [ ]:
oil['date'] = pd.to_datetime(oil['date'])
print(f"Oil από: {oil['date'].min()} έως {oil['date'].max()}")
print(f"Train από: {train['date'].min()} έως {train['date'].max()}")
print(f"\nΣτατιστικά τιμής πετρελαίου:")
print(oil['dcoilwtico'].describe())

In [ ]:
print(f"Shape: {holidays.shape}")
print(f"Columns: {list(holidays.columns)}")
print(f"Missing: {holidays.isnull().sum().sum()}")
print(f"\nFirst rows:")
holidays.head(10)

In [ ]:
for col in ['type', 'locale', 'locale_name', 'transferred']:
    print(f"\n{col}: {holidays[col].nunique()} μοναδικές")
    print(holidays[col].value_counts())

In [ ]:
print(f"Shape: {transactions.shape}")
print(f"Columns: {list(transactions.columns)}")
print(f"Missing: {transactions.isnull().sum().sum()}")
print(f"\nFirst rows:")
transactions.head(10)
print(f"\nΣτατιστικά:")
transactions['transactions'].describe()